# Package

In [62]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import folium
import time

import matplotlib
import matplotlib.cm as cm
import missingno as msno
import matplotlib.colors as mcolors

import geopandas as gpd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm
from shapely.geometry import Point

from sklearn.preprocessing import StandardScaler

import random
from sklearn.model_selection import StratifiedKFold, GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.metrics import (
    f1_score, recall_score, precision_score, roc_auc_score,
    accuracy_score, confusion_matrix, classification_report, make_scorer, precision_recall_curve
)
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from imblearn.ensemble import EasyEnsembleClassifier


from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTETomek
from imblearn.under_sampling import TomekLinks
from imblearn.pipeline import Pipeline

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

from matplotlib import font_manager, rc
import os
import platform
import shutil
from pathlib import Path
import sys, io
import warnings



# Function

In [ ]:
# ============================================================
# 0) 필수 import
# ============================================================
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")


# ============================================================
# 1) VS Code / 로컬에서 한글 깨짐 방지 설정 (matplotlib + pandas)
#    - Windows: 맑은 고딕 (Malgun Gothic)
# ============================================================
def set_korean_env():
    system = platform.system()

    # matplotlib 한글 폰트
    if system == "Windows":
        plt.rcParams["font.family"] = "Malgun Gothic"


    # 마이너스(−) 깨짐 방지
    plt.rcParams["axes.unicode_minus"] = False

    # pandas 출력 옵션 (한글/가독성)
    pd.set_option("display.max_columns", 200)
    pd.set_option("display.width", 200)
    pd.set_option("display.max_colwidth", 200)

set_korean_env()


# ============================================================
# 2) "프로젝트 루트 탐색" + data 폴더 자동 연결
# ============================================================
def find_project_root(start: Path, max_up: int = 8) -> Path:
    """
    start 위치에서 위로 올라가며 'data' 폴더를 찾고,
    그 안에 big_data_set1_f 같은 핵심 파일이 있으면 프로젝트 루트로 확정.
    """
    cur = start.resolve()
    for _ in range(max_up):
        data_dir = cur / "data"
        if data_dir.exists():
            # 핵심 파일이 존재하면 프로젝트 루트로 인정
            if list(data_dir.glob("big_data_set1_f*.csv")) or list(data_dir.glob("big_data_set1_f*.CSV")):
                return cur
        cur = cur.parent
    raise FileNotFoundError("프로젝트 루트를 찾지 못했습니다. 'DM_closure-prediction' 폴더에서 실행 중인지 확인하세요.")

# 노트북/VSCode 실행 위치 기반으로 루트 찾기
BASE_DIR = Path.cwd()
PROJECT_ROOT = find_project_root(BASE_DIR)
DATA_DIR = PROJECT_ROOT / "data"

print("BASE_DIR      :", BASE_DIR)
print("PROJECT_ROOT  :", PROJECT_ROOT)
print("DATA_DIR      :", DATA_DIR)
print("DATA_DIR files:", [p.name for p in DATA_DIR.iterdir()])


# ============================================================
# 3) 파일명(한글 정규화/띄어쓰기/확장자 숨김) 때문에 FileNotFoundError가 계속 나는 문제 해결
#    -> data 폴더에서 "실제 존재하는 파일"을 glob로 찾아서 경로 지정
# ============================================================
def pick_one(patterns, desc: str) -> Path:
    candidates = []
    for pat in patterns:
        candidates.extend(list(DATA_DIR.glob(pat)))

    # 중복 제거(순서 유지)
    unique = []
    seen = set()
    for c in candidates:
        if c.as_posix() not in seen:
            unique.append(c)
            seen.add(c.as_posix())

    if not unique:
        raise FileNotFoundError(f"❌ {desc} 파일을 찾지 못했습니다.\n"
                                f"   patterns={patterns}\n"
                                f"   DATA_DIR={DATA_DIR}\n"
                                f"   DATA_DIR files={[p.name for p in DATA_DIR.iterdir()]}")
    if len(unique) > 1:
        print(f" {desc} 후보가 여러 개라 첫 번째를 사용합니다:", [u.name for u in unique])
    return unique[0]


# --- (캡처 기준 파일명에 맞춘 패턴) ---
path1 = pick_one(["big_data_set1_f*.csv", "big_data_set1_f*.CSV"], "big_data_set1_f")
path2 = pick_one(["big_data_set2_f*.csv", "big_data_set2_f*.CSV"], "big_data_set2_f")
path3 = pick_one(["big_data_set3_f*.csv", "big_data_set3_f*.CSV"], "big_data_set3_f")

# shp (서울시 상권분석서비스(영역-상권).shp)
shp_path = pick_one(["*영역-상권*.shp", "*.shp"], "상권영역 Shapefile(.shp)")

# 주소-좌표
address_xy_path = pick_one(["*가맹점주소*좌표*.csv", "*경도위도*좌표*.csv", "*주소*좌표*.csv"], "가맹점 주소-좌표")

# 상권분석정보 (서울시 상권분석서비스(길단위인구-상권).csv)
trade_area_path = pick_one(["*길단위인구*상권*.csv", "*길단위*상권*.csv", "*상권분석서비스*길단위*.csv"], "상권분석정보")

# 생활물가지수 (생활물가지수_20251005164822.csv)
consumer_price_index_path = pick_one(["*생활물가지수*.csv", "*물가지수*.csv", "*CPI*.csv"], "생활물가지수(CPI)")

print("\n Paths resolved")
print(" - path1:", path1.name)
print(" - path2:", path2.name)
print(" - path3:", path3.name)
print(" - shp  :", shp_path.name)
print(" - addr :", address_xy_path.name)
print(" - trade:", trade_area_path.name)
print(" - cpi  :", consumer_price_index_path.name)


# ============================================================
# 4) CSV 인코딩 자동 처리 (utf-8 / cp949 / euc-kr)
# ============================================================
def read_csv_safely(path: Path):
    for enc in ["utf-8", "utf-8-sig", "cp949", "euc-kr"]:
        try:
            return pd.read_csv(path, encoding=enc), enc
        except UnicodeDecodeError:
            pass
    # 마지막 보험(한글 깨질 수 있음)
    return pd.read_csv(path, encoding="latin1"), "latin1"


# ============================================================
# 5) Data Load & 컬럼명 변경 (네가 쓰던 rename 그대로 유지)
# ============================================================
def data_load(path1, path2, path3, shp_path, address_xy_path, trade_area_path, consumer_price_index_path):
    # CSV
    big_data_set1_f, enc1 = read_csv_safely(path1)
    big_data_set2_f, enc2 = read_csv_safely(path2)
    big_data_set3_f, enc3 = read_csv_safely(path3)

    unique_addresses, enc_addr = read_csv_safely(address_xy_path)
    trade_area_info, enc_trade = read_csv_safely(trade_area_path)
    consumer_price_index, enc_cpi = read_csv_safely(consumer_price_index_path)

    print(f"\n encodings | ds1={enc1}, ds2={enc2}, ds3={enc3}, address={enc_addr}, trade={enc_trade}, cpi={enc_cpi}")

    # SHP (shx/dbf가 같은 폴더에 있어야 함)
    # (캡처상 있으니 정상일 것)
    gdf = gpd.read_file(shp_path)

    # rename (네 코드 그대로)
    big_data_set1_f = big_data_set1_f.rename(columns={
        'ENCODED_MCT': '가맹점구분번호',
        'MCT_BSE_AR': '가맹점주소',
        'MCT_NM': '가맹점명',
        'MCT_BRD_NUM': '브랜드구분코드',
        'MCT_SIGUNGU_NM': '가맹점지역',
        'HPSN_MCT_ZCD_NM': '업종',
        'HPSN_MCT_BZN_CD_NM': '상권',
        'ARE_D': '개설일',
        'MCT_ME_D': '폐업일'
    })

    big_data_set2_f = big_data_set2_f.rename(columns={
        'ENCODED_MCT': '가맹점구분번호',
        'TA_YM': '기준년월',
        'MCT_OPE_MS_CN': '가맹점 운영개월수 구간',
        'RC_M1_SAA': '매출금액 구간',
        'RC_M1_TO_UE_CT': '매출건수 구간',
        'RC_M1_UE_CUS_CN': '유니크 고객 수 구간',
        'RC_M1_AV_NP_AT': '객단가 구간',
        'APV_CE_RAT': '취소율 구간',
        'DLV_SAA_RAT': '배달매출금액 비율',
        'M1_SME_RY_SAA_RAT': '동일 업종 매출금액 비율',
        'M1_SME_RY_CNT_RAT': '동일 업종 매출건수 비율',
        'M12_SME_RY_SAA_PCE_RT': '동일 업종 내 매출 순위 비율',
        'M12_SME_BZN_SAA_PCE_RT': '동일 상권 내 매출 순위 비율',
        'M12_SME_RY_ME_MCT_RAT': '동일 업종 내 해지 가맹점 비중',
        'M12_SME_BZN_ME_MCT_RAT': '동일 상권 내 해지 가맹점 비중'
    })

    big_data_set3_f = big_data_set3_f.rename(columns={
        'ENCODED_MCT': '가맹점구분번호',
        'TA_YM': '기준년월',
        'M12_MAL_1020_RAT': '남성 20대이하 고객 비중',
        'M12_MAL_30_RAT': '남성 30대 고객 비중',
        'M12_MAL_40_RAT': '남성 40대 고객 비중',
        'M12_MAL_50_RAT': '남성 50대 고객 비중',
        'M12_MAL_60_RAT': '남성 60대이상 고객 비중',
        'M12_FME_1020_RAT': '여성 20대이하 고객 비중',
        'M12_FME_30_RAT': '여성 30대 고객 비중',
        'M12_FME_40_RAT': '여성 40대 고객 비중',
        'M12_FME_50_RAT': '여성 50대 고객 비중',
        'M12_FME_60_RAT': '여성 60대이상 고객 비중',
        'MCT_UE_CLN_REU_RAT': '재방문 고객 비중',
        'MCT_UE_CLN_NEW_RAT': '신규 고객 비중',
        'RC_M1_SHC_RSD_UE_CLN_RAT': '거주 이용 고객 비율',
        'RC_M1_SHC_WP_UE_CLN_RAT': '직장 이용 고객 비율',
        'RC_M1_SHC_FLP_UE_CLN_RAT': '유동인구 이용 고객 비율'
    })

    return big_data_set1_f, big_data_set2_f, big_data_set3_f, gdf, unique_addresses, trade_area_info, consumer_price_index


# ============================================================
# 6) 실행 (로드 확인)
# ============================================================
big_data_set1_f, big_data_set2_f, big_data_set3_f, gdf, unique_addresses, trade_area_info, consumer_price_index = data_load(
    path1, path2, path3, shp_path, address_xy_path, trade_area_path, consumer_price_index_path
)

print("\n Loaded shapes")
print(" - big_data_set1_f:", big_data_set1_f.shape)
print(" - big_data_set2_f:", big_data_set2_f.shape)
print(" - big_data_set3_f:", big_data_set3_f.shape)
print(" - gdf:", gdf.shape)
print(" - unique_addresses:", unique_addresses.shape)
print(" - trade_area_info:", trade_area_info.shape)
print(" - consumer_price_index:", consumer_price_index.shape)


✅ BASE_DIR      : c:\Users\관리자\Desktop\이정수꺼\공모전\DM_develop\DM_closure-prediction\src
✅ PROJECT_ROOT  : C:\Users\관리자\Desktop\이정수꺼\공모전\DM_develop\DM_closure-prediction
✅ DATA_DIR      : C:\Users\관리자\Desktop\이정수꺼\공모전\DM_develop\DM_closure-prediction\data
✅ DATA_DIR files: ['all_data_v1_update.csv', 'big_data_set1_f.csv', 'big_data_set2_f.csv', 'big_data_set3_f.csv', '가맹점주소_경도위도_좌표.csv', '빅콘테스트 데이터 레이아웃.xlsx', '생활물가지수_20251005164822.csv', '서울시 상권분석서비스(길단위인구-상권).csv', '서울시 상권분석서비스(영역-상권).cpg', '서울시 상권분석서비스(영역-상권).dbf', '서울시 상권분석서비스(영역-상권).prj', '서울시 상권분석서비스(영역-상권).shp', '서울시 상권분석서비스(영역-상권).shx']

✅ Paths resolved
 - path1: big_data_set1_f.csv
 - path2: big_data_set2_f.csv
 - path3: big_data_set3_f.csv
 - shp  : 서울시 상권분석서비스(영역-상권).shp
 - addr : 가맹점주소_경도위도_좌표.csv
 - trade: 서울시 상권분석서비스(길단위인구-상권).csv
 - cpi  : 생활물가지수_20251005164822.csv

✅ encodings | ds1=cp949, ds2=cp949, ds3=utf-8, address=utf-8, trade=cp949, cpi=utf-8

✅ Loaded shapes
 - big_data_set1_f: (4185, 9)
 - big_data_set2_f: (86590, 

## Preprocessing

In [69]:
# Preprocessing
def Preprocessing(big_data_set1_f, gdf, unique_addresses, big_data_set2_f, big_data_set3_f):
  # ======================================================
  #  DATASET 1 PREPROCESSING (가맹점 기본 정보 정제)
  # ======================================================

  # 1) 폐업일·개설일 → datetime 형 변환
  big_data_set1_f["폐업일"] = pd.to_datetime(big_data_set1_f["폐업일"], format="%Y%m%d")
  big_data_set1_f["개설일"] = pd.to_datetime(big_data_set1_f["개설일"], format="%Y%m%d")

  # 2) 폐업여부 컬럼 생성 (Boolean Feature Engineering)
  big_data_set1_f["폐업여부"] = big_data_set1_f["폐업일"].notna()

  # 3) 주소 표기 정규화 (마침표 제거 & '서울' → '서울특별시' 통일)
  def update_address(addr):
      if isinstance(addr, str):
          if addr.endswith('.'):  # 잘못된 OCR 및 데이터 입력 정리
              addr = addr[:-1]
          # '서울' 표기를 '서울특별시'로 통일(행정구역 기준 일치 목적)
          parts = addr.split()
          if parts and parts[0] == '서울':
              parts[0] = '서울특별시'
              return ' '.join(parts)
      return addr

  big_data_set1_f['가맹점주소'] = big_data_set1_f['가맹점주소'].apply(update_address)

  # 4) 성동구가 아닌 가맹점 제거
  outside_seongdong = big_data_set1_f[~big_data_set1_f['가맹점주소'].str.contains('성동구', na=False)]
  big_data_set1_f.drop(outside_seongdong.index, inplace=True)

  # 5) 신한카드 상권 데이터 기반 1차 상권 결측치 대체 수행
  #    - 상권이 존재하는 레코드만 사용 (근거 데이터 확실화)
  mapping_src = (
      big_data_set1_f.dropna(subset=['상권'])               # 유효한 상권 정보만 추출
                      .drop_duplicates(subset=['가맹점주소'])  # 동일주소 중복 제거
                      .set_index('가맹점주소')['상권']
  )

  #    - 상권 결측치 주소 매핑 수행
  big_data_set1_f['상권'] = big_data_set1_f.apply(
      lambda row: mapping_src.get(row['가맹점주소'], row['상권']),
      axis=1
  )

  # 6) 특정 오기입 상권 보정 (도메인 검증 기반 핸드픽 수정)
  big_data_set1_f.loc[big_data_set1_f['상권'] == '서면역', '상권'] = '성수'
  big_data_set1_f.loc[big_data_set1_f['상권'] == '건대입구', '상권'] = '뚝섬'
  big_data_set1_f.loc[big_data_set1_f['상권'] == '동대문역사문화공원역', '상권'] = '금남시장'
  big_data_set1_f.loc[big_data_set1_f['상권'] == '압구정로데오', '상권'] = '금남시장'
  big_data_set1_f.loc[big_data_set1_f['상권'] == '오남', '상권'] = '한양대'
  big_data_set1_f.loc[big_data_set1_f['상권'] == '방배역', '상권'] = '성수'
  big_data_set1_f.loc[big_data_set1_f['상권'] == '미아사거리', '상권'] = '성수'
  big_data_set1_f.loc[big_data_set1_f['상권'] == '풍산지구', '상권'] = '뚝섬'
  big_data_set1_f.loc[big_data_set1_f['상권'] == '자양', '상권'] = '성수'

  # ========================================================
  #  서울시 상권분석서비스(영역-상권).shp 활용한 상권 매핑
  # 1) 주소 → 좌표 변환 (사전 계산 결과 사용)
  # 2) 좌표 → 상권 영역 매핑 (공공데이터 shapefile 기준)
  # ========================================================

  # 1) 주소 → 좌표 변환
    # 상권 좌표 계산 과정은 외부 API 호출(Nominatim)로
    # 총 2,434개 주소 기준 수십 분 이상 소요됩니다.
    # → reproducibility 및 성능을 위해 결과를 CSV로 저장해 불러오는 방식을 사용.
    #   (해당 제출 코드에서는 geocoding 수행 코드는 주석 처리하고 결과만 반영)
  '''
  geolocator = Nominatim(user_agent="store_mapper", timeout=10)
  geocode = RateLimiter(geolocator.geocode, min_delay_seconds=2)
  tqdm.pandas()
  unique_addresses = pd.DataFrame(big_data_set1_f["가맹점주소"].drop_duplicates().reset_index(drop=True)) #2434개의 주소
  unique_addresses["location"] = unique_addresses["가맹점주소"].progress_apply(geocode)
  unique_addresses["lat"] = unique_addresses["location"].apply(lambda loc: loc.latitude if loc is not None else None)
  unique_addresses["lon"] = unique_addresses["location"].apply(lambda loc: loc.longitude if loc is not None else None)
  unique_addresses.to_csv("/content/drive/MyDrive/[LG 8기] HHGG 공모전/0_data/가맹점주소_좌표_2.csv", index=False)
  '''
  #    - shapefile 기반 상권 매핑
  gdf_sd = gdf[gdf['SIGNGU_CD'] == '11200'].copy() #성동구
  if gdf_sd.crs != "EPSG:4326":
      gdf_sd = gdf_sd.to_crs(epsg=4326)

  #    - 좌표 누락 주소 수동보완
  manual_coords = {
    '서울특별시 성동구 하왕십리1동': (37.564627, 127.028785),
    '서울특별시 성동구 동호로5길 131': (37.546917, 127.010291),
    '서울특별시 성동구 성수일로3길 7-7': (37.542166, 127.047935),
    '서울특별시 성동구 왕십리로26길 2-1': (37.563821, 127.033015),
    '서울특별시 성동구 성수일로3길 12': (37.542614, 127.048009),
    '서울특별시 성동구 성수일로3길 5-7': (37.542034, 127.048151),
    '서울특별시 성동구 성수일로3길 7-1': (37.542191, 127.047930),
    '서울특별시 성동구 마장로37길 40 (마장동)': (37.569016, 127.041038),
    '서울특별시 성동구 성수일로3길 4-13': (37.542498, 127.048248),
    '서울특별시 성동구 성수일로3길 5-12': (37.541983, 127.047959),
    '서울특별시 성동구 왕십리로 363-1 (하왕십리동)': (37.564073, 127.029584),
    '서울특별시 성동구 옥수2동': (37.545731, 127.020355),
    '서울특별시 성동구 하왕십리2동': (37.561935, 127.029221),
    '서울특별시 성동구 성수일로3길 2': (37.543088, 127.047050),
    '서울특별시 성동구 왕십리로26길 30': (37.562304, 127.031548),
    '서울특별시 성동구 왕십리로26길 6': (37.563539, 127.032152),
    '서울특별시 성동구 성수동2가 289-175': (37.545890, 127.049439),
    '서울특별시 성동구 둘레9나길 7': (37.548487, 127.042531),
    '서울특별시 성동구 왕십리로26길 8': (37.563402, 127.031899),
    '서울특별시 성동구 왕십리로26길 11': (37.563177, 127.032128),
    '서울특별시 성동구 왕십리로26길 32': (37.562234, 127.031269),
    '서울특별시 성동구 왕십리로 지하300': (37.561332, 127.037435),
    '서울특별시 성동구 왕십리로26길 7-2': (37.563467, 127.032646),
    '서울특별시 성동구 천호대로 지하300': (37.568436, 127.045233)
    }
  for addr, (lat, lon) in manual_coords.items():
      unique_addresses.loc[unique_addresses['가맹점주소'] == addr, ['lat', 'lon']] = [lat, lon]

  # 2) 좌표 → 상권 영역 매핑
  #    - 서울 열린데이터 기반 결측 상권 보정
  unique_addresses["point"] = unique_addresses.apply(lambda row: Point(row["lon"], row["lat"]), axis=1)
  def find_containing_area(point, gdf_polygons):
      for _, row in gdf_polygons.iterrows():
          if row["geometry"].contains(point):
              return row["TRDAR_CD_N"]
      return None
  unique_addresses['상권'] = unique_addresses['point'].apply(lambda p: find_containing_area(p, gdf_sd))
  seongdong_area = unique_addresses['상권'].value_counts().reset_index()['상권'].tolist()

  #    - 신한카드 데이터 기반 결측 상권 보정
  address_to_district = (
    big_data_set1_f.dropna(subset=['상권'])
    .drop_duplicates(subset=['가맹점주소'])
    .set_index('가맹점주소')['상권'])
  unique_addresses['상권'] = unique_addresses['상권'].fillna(
      unique_addresses['가맹점주소'].map(address_to_district))

  #    - 상권 포함 여부 컬럼 추가
  unique_addresses['상권포함여부'] = unique_addresses['상권'].notnull()

  #    - 상권 외 지역 처리
  unique_addresses['상권'].fillna('(상권 외 지역)', inplace=True)

  #    - 원본 데이터 반영
  address_to_district_map = unique_addresses.set_index('가맹점주소')['상권']
  big_data_set1_f['상권'] = big_data_set1_f['가맹점주소'].map(address_to_district_map)

  # ========================================================
  #  상권 지도 시각화
  # ========================================================

  # 1) 지도 생성
  map = folium.Map(location=[37.563, 127.036], zoom_start=14)

  # 2) 상권별 고유 색상 지정
  unique_codes = gdf_sd['TRDAR_CD'].unique()
  cmap = cm.get_cmap('tab20', len(unique_codes))
  color_map = {code: mcolors.to_hex(cmap(i)) for i, code in enumerate(unique_codes)}
  gdf_sd["color"] = gdf_sd["TRDAR_CD"].map(color_map)

  # 3) 상권 경계 표시
  for _, row in gdf_sd.iterrows():
      name = row["TRDAR_CD_N"]
      color = row["color"]
      folium.GeoJson(
          row["geometry"],
          style_function=lambda feature, color=color: {
              "fillColor": color,
              "color": "black",
              "weight": 1,
              "fillOpacity": 0.4,
          },
          tooltip=folium.Tooltip(f"상권: {name}", sticky=True)
      ).add_to(map)

  # 4) 가맹점 마커 표시 (상권 포함 여부에 따라 색상 다르게)
  for _, row in unique_addresses.dropna(subset=["lat", "lon"]).iterrows():
      color = "red" if row["상권포함여부"] is True else "blue"
      folium.Marker(
          location=[row["lat"], row["lon"]],
          popup=row["가맹점주소"],  #마커를 클릭시 주소 팝업
          tooltip="상권 있음" if color == "red" else "상권 외부",
          icon=folium.Icon(color=color, icon="info-sign")
      ).add_to(map)

  # ======================================================
  #  DATASET 2 PREPROCESSING (가맹점 매출 정보 정제)
  # ======================================================

  # 1) 결측치 처리 (-999999.9 → NaN)
  big_data_set2_f.replace(-999999.9, np.nan, inplace=True)

  # 2) 구간형 컬럼 숫자형으로 변환
  bucket_cols = ['가맹점 운영개월수 구간', '매출금액 구간', '매출건수 구간',
                  '유니크 고객 수 구간', '객단가 구간', '취소율 구간']
  for col in bucket_cols:
      if col in big_data_set2_f.columns:
        big_data_set2_f[col] = pd.to_numeric(
            big_data_set2_f[col].astype(str).str.split('_').str[0],
            errors='coerce'
        ).astype('Int64')

  # 3) 기준년월 datetime 변환
  big_data_set2_f["기준년월"] = pd.to_datetime(big_data_set2_f["기준년월"], format = "%Y%m")

  # 4) 배달매출금액 비율 결측치 처리 함수
  def impute_delivery_sales_ratio(df):
    """
    - 모든 기준년월 결측: 0으로 대체
    - 일부 결측: 가맹점별 평균으로 대체
    """
    #    -가맹점별 전체 기준년월 수
    total_cnt = df.groupby("가맹점구분번호")["기준년월"].nunique().reset_index(name="전체 기준년월 수")
    #    -결측 기준년월 수
    missing_cnt = df[df['배달매출금액 비율'].isna()] \
        .groupby("가맹점구분번호")["기준년월"].nunique().reset_index(name="결측 기준년월 수")
    result = total_cnt.merge(missing_cnt, on="가맹점구분번호", how="left").fillna(0)
    result["결측 비율(%)"] = (result["결측 기준년월 수"] / result["전체 기준년월 수"]) * 100
    #    -100% 결측 가맹점은 0으로
    missing_100_list = result[result["결측 비율(%)"] == 100]["가맹점구분번호"].tolist()
    df.loc[
        (df["가맹점구분번호"].isin(missing_100_list)) & (df["배달매출금액 비율"].isna()),
        "배달매출금액 비율"
    ] = 0
    #    -일부 결측은 가맹점별 평균으로 대체
    mean_by_store = df.groupby("가맹점구분번호")["배달매출금액 비율"].transform("mean")
    df["배달매출금액 비율"].fillna(mean_by_store, inplace=True)

    return df

  big_data_set2_f = impute_delivery_sales_ratio(big_data_set2_f)

  # 5) 취소율 구간 결측치 처리(가맹점별 평균으로)
  big_data_set2_f["취소율 구간"] = big_data_set2_f["취소율 구간"].astype(float)
  big_data_set2_f["취소율 구간"] = big_data_set2_f.groupby("가맹점구분번호")["취소율 구간"].transform(
      lambda x: x.fillna(0)
  )


  # ======================================================
  #  DATASET 3 PREPROCESSING (가맹점 고객 정보 정제)
  # ======================================================

  # 1) 결측치 처리 (-999999.9 → NaN)
  big_data_set3_f.replace(-999999.9, np.nan, inplace=True)

  # 2) 기준년월(datetime) 변환 및 정렬
  big_data_set3_f["기준년월"] = pd.to_datetime(big_data_set3_f["기준년월"], format = "%Y%m")
  big_data_set3_f.sort_values(by=['가맹점구분번호', '기준년월'], inplace=True)

  # 3) 고객 비중 관련 컬럼 결측치 처리 (그룹 최빈값 → 전체 최빈값)
  customer_ratio_cols = [
      "남성 20대이하 고객 비중", "남성 30대 고객 비중", "남성 40대 고객 비중",
      "남성 50대 고객 비중", "남성 60대이상 고객 비중", "여성 20대이하 고객 비중",
      "여성 30대 고객 비중", "여성 40대 고객 비중", "여성 50대 고객 비중",
      "여성 60대이상 고객 비중", "재방문 고객 비중", "신규 고객 비중",
      "거주 이용 고객 비율", "직장 이용 고객 비율", "유동인구 이용 고객 비율"
  ]
  for col in customer_ratio_cols:
    # 그룹 내 최빈값으로 대체
    mode_by_group = big_data_set3_f.groupby('가맹점구분번호')[col].transform(
        lambda x: x.mode()[0] if not x.mode().empty else np.nan
        )
    big_data_set3_f[col] = big_data_set3_f[col].fillna(mode_by_group)

    # 남은 결측치는 전체 데이터의 최빈값으로 대체
    global_mode = big_data_set3_f[col].mode()
    if not global_mode.empty:
      big_data_set3_f[col] = big_data_set3_f[col].fillna(global_mode[0])

  # ======================================================
  #  [ DATASET 1 + DATASET 2 + DATASET 3 ] 병합
  # ======================================================

  # 1) 필요한 컬럼만 추출
  store_info_cols  = ['가맹점구분번호',	'가맹점주소',	'업종',	'상권',	'개설일',	'폐업일','폐업여부']
  big_data_set1_f = big_data_set1_f[store_info_cols]

  # 2) big_data_set1 + big_data_set2 + big_data_set3 병합
  merged_1and2 = pd.merge(
    big_data_set1_f,
    big_data_set2_f,
    on="가맹점구분번호",
    how="left"
    )

  merged_1and2and3 = pd.merge(
      merged_1and2,
      big_data_set3_f,
      on=["가맹점구분번호", "기준년월"],
      how="left"
  )

  # ======================================================
  #  월별 매출 데이터 7개 이하 가맹점 제외(학습 품질을 떨어뜨림)
  # ======================================================
  # 1) 데이터 개수 집계
  monthly_counts = (
      merged_1and2and3
      .groupby('가맹점구분번호')
      .agg(
          데이터개수=('가맹점구분번호', 'size'),
          폐업여부=('폐업여부', 'first')
      )
      .reset_index()
  )

  # 2) 1~7개 이하 제외
  valid_stores = monthly_counts.loc[
      ~monthly_counts['데이터개수'].between(1, 7),
      '가맹점구분번호'
  ]

  # 3) merged_1and2and3에서 해당 가맹점만 유지
  merged_1and2and3 = merged_1and2and3[
      merged_1and2and3['가맹점구분번호'].isin(valid_stores)
  ]
  return merged_1and2and3, address_to_district_map, seongdong_area, map


## 파생변수 생성

In [70]:
def create_features(path1, merged_1and2and3, trade_area_info, seongdong_area, consumer_price_index):

  # ======================================================
  # 상권 집객력(feature) 생성
  # ======================================================

  # 1) 외부데이터의 상권별 평균 유동인구 계산
  #    - 23~24년 정보 필터링
  target_quarters = [20231, 20232, 20233, 20234, 20241, 20242, 20243, 20244]
  trade_area_filtered = trade_area_info[
      trade_area_info['기준_년분기_코드'].isin(target_quarters)
      ]
  #    - 외부 데이터의 성동구 상권만 추출
  population_avg = (
      trade_area_filtered
      .groupby("상권_코드_명", as_index=False)["총_유동인구_수"]
      .mean()
      .rename(columns={"총_유동인구_수":"상권_집객력"})
  )

  trade_area_population = population_avg[
      population_avg["상권_코드_명"].isin(seongdong_area)
  ].copy()

  # 2) 신한카드 데이터의 상권 유동인구 평균 계산
  #    - 외부 데이터 상권 이름으로 임시 치환
  raw_big_data_set1_f = pd.read_csv(path1, encoding='cp949') #dataset1\
  raw_big_data_set1_f = raw_big_data_set1_f.rename(columns={'HPSN_MCT_BZN_CD_NM': '상권'})
  raw_big_data_set1_f["상권"] = raw_big_data_set1_f["상권"].replace({
      "화양시장": "화양",
      "마장동": "마장",
      "장한평자동차": "장한평"
  })
  #    - 상권명을 외부 데이터 상권 이름과 매칭
  def find_match(name):
      name = str(name)
      for short in raw_big_data_set1_f["상권"].dropna().astype(str):
          if short in name or name in short:
              return short
          elif short[:2] in name or name[:2] in short:
              return short
      return None
  trade_area_population["상권"] = trade_area_population["상권_코드_명"].apply(find_match)
  trade_area_population = trade_area_population.reset_index(drop=True)

  #    - 장한평역 데이터 예외 처리 및 이름 치환
  jang_pop = population_avg[population_avg['상권_코드_명'] == '장한평역(장한평)'].copy()
  jang_pop['상권'] = '장한평자동차'

  #    - 병합
  shinhan_trade_area_population =(trade_area_population[['상권', '상권_집객력']]
                                  .groupby('상권', as_index=False)
                                  .mean())
  shinhan_trade_area_population = pd.concat([shinhan_trade_area_population, jang_pop[['상권', '상권_집객력']]], ignore_index=True)

  # 3) 상권 외 지역의 집객력
  #    - 전체 상권의 하위 25%로 계산
  q1 = population_avg['상권_집객력'].quantile(0.25)

  # 4) 최종 상권 집객력 정리
  #    - 기존 '상권' 컬럼 드랍 후, '상권_코드_명'을 '상권'으로 이름 변경
  trade_area_population = (
      trade_area_population
        .drop(columns=['상권'], errors='ignore')
        .rename(columns={'상권_코드_명': '상권'})
  )
  #    - 최종 합침(area_active)
  area_active = pd.concat(
      [   trade_area_population,
          shinhan_trade_area_population,
          pd.DataFrame([{'상권': '(상권 외 지역)', '상권_집객력': q1}])
      ],ignore_index=True
    ).drop_duplicates(subset=['상권'])

  #    - 마장 -> 마장동
  area_active['상권'] = area_active['상권'].replace('마장', '마장동')

  # ======================================================
  # 상권 소비력(feature) 생성
  # ======================================================
  # 1) 상권별 평균값 집계
  area_agg = (
      merged_1and2and3.groupby("상권", as_index=False)
          .agg({
              "매출금액 구간": "mean",
              "유니크 고객 수 구간": "mean",
              "동일 업종 내 매출 순위 비율": "mean"  # 0일수록 상위
          })
  )

  # # 2) 순위/구간 점수화
  #    - 매출금액, 고객 수 구간: 1~6 → 0~100 선형 변환
  area_agg["매출_점수"] = (6 - area_agg["매출금액 구간"]) / 5 * 100
  # area_agg["객단가_점수"] = (6 - area_agg["객단가 구간"]) / 5 * 100
  area_agg["고객수_점수"] = (6 - area_agg["유니크 고객 수 구간"]) / 5 * 100
  #    - 업종 내 순위: 0 이 상위 → 역변환
  area_agg["업종내순위_역점수"] = 100 - area_agg["동일 업종 내 매출 순위 비율"]

  # 3) 상권 소비력 계산
  #    - 매출, 고객 수, 업종 내 순위 점수 평균

  sel_col = ["매출_점수", #"객단가_점수",
            "고객수_점수", "업종내순위_역점수"]
  area_agg["상권_소비력"] = area_agg[sel_col].mean(axis=1)
  area_active= area_active.merge(area_agg[["상권","상권_소비력"]])

  # ======================================================
  # 상권 활성화 지수(feature) 생성
  #  : 집객력 & 소비력 기하평균
  # ======================================================
  # 1) 표준화
  scaler = StandardScaler()
  scaled = scaler.fit_transform(area_active[["상권_집객력", "상권_소비력"]])

  # 2) 음수값 처리 및 기하평균 계산
  scaled_shifted = scaled - scaled.min(axis=0) + 1e-9
  area_active["상권_활성화지수"] = np.sqrt(scaled_shifted[:, 0] * scaled_shifted[:, 1])

  # 3) 가맹점 기준으로 상권 활성화지수(+집객력, +소비력) 추가
  create_features_result = merged_1and2and3.drop_duplicates('가맹점구분번호')[['가맹점구분번호','상권','폐업여부']]
  create_features_result = create_features_result.merge(area_active, on="상권", how="left").drop(columns='상권')
  create_features_result

  # ======================================================
  # 가맹점별 물가 대비 민감도 (feature) 생성
  #     - 생활물가지수 외부 데이터 활용
  #     - 매출액_구간_변화율 / 생활물가지수_변화율
  # ======================================================

  # 1) 매출액_구간_변화율
  #     - 전월 대비 매출액 구간 변동 계산
  #       * diff(): 현재월 - 전월
  #       * -1 : 매출 감소를 '성장'으로 표현하고 싶을 때 방향 반전
  merged_1and2and3 = merged_1and2and3.sort_values(by=['가맹점구분번호', '기준년월'])
  merged_1and2and3['매출액_구간변동'] = (
    merged_1and2and3.groupby('가맹점구분번호')['매출금액 구간']
    .diff() * -1
  )
  #     - 첫 달의 NaN(비교 불가능한 첫 달)은 0으로 채우기
  merged_1and2and3['매출액_구간변동'].fillna(0, inplace=True)

  #     - 구간 정보를 바탕으로 단계별 변화율 계산
  total_steps = 7 - 1  # 1~7 구간이므로 총 6단계
  change_rate_per_step = 100 / total_steps  # 약 16.67%
  merged_1and2and3['매출액_구간_변화율'] = merged_1and2and3['매출액_구간변동'] * change_rate_per_step / 100

  # 2) 생활물가지수_변화율
  consumer_price_index['생활물가지수_변화율'] = consumer_price_index['생활물가지수'].pct_change()
  consumer_price_index['시점'] = pd.to_datetime(consumer_price_index['시점'], format='%Y. %m')

  merged_1and2and3 = merged_1and2and3.merge(
    consumer_price_index[['시점', '생활물가지수_변화율']],
    left_on='기준년월',
    right_on='시점',
    how='left'
  )

  # 3) 물가 대비 민감도 계산 & 추가
  merged_1and2and3['물가 대비 민감도'] = merged_1and2and3['매출액_구간_변화율'] / merged_1and2and3['생활물가지수_변화율']
  average_sensitivity_by_store = merged_1and2and3.groupby('가맹점구분번호')['물가 대비 민감도'].mean().reset_index()
  create_features_result = create_features_result.merge(
    average_sensitivity_by_store,
    on='가맹점구분번호',
    how='left'
  )

  # ======================================================
  # 가맹점별 소비자 구매력(feature) 생성
  # ======================================================
  # 1) 가맹점별 평균값 집계
  #    - '매출금액 구간', '유니크 고객 수 구간'의 평균
  store_agg = (
      merged_1and2and3.groupby("가맹점구분번호", as_index=False)
          .agg({
              "매출금액 구간": "mean",
              "유니크 고객 수 구간": "mean"
          })
  )
  # 2) 점수화
  #    - 구간이 높을수록 낮은 점수 부여 (1~6 구간 → 100~0 점수)
  store_agg["매출_점수"] = (6 - store_agg["매출금액 구간"]) / 5 * 100
  store_agg["고객수_점수"] = (6 - store_agg["유니크 고객 수 구간"]) / 5 * 100

  # 3) 구매력지수 계산 (가중치는 상황에 맞게 조정 가능)
  store_col = ["매출_점수", "고객수_점수"]
  store_agg["소비자 구매력"] = store_agg[store_col].mean(axis=1)

  # 4) 기존 데이터에 병합
  #    - '가맹점구분번호' 기준으로 소비자 구매력 추가
  create_features_result = create_features_result.merge(
      store_agg[["가맹점구분번호", "소비자 구매력"]],
      on="가맹점구분번호",
      how="left")

  # ======================================================
  # 가맹점 유지기간(feature) 생성
  # ======================================================

  # 유지기간 계산
  period = merged_1and2and3.copy()
  period["개설일"] = pd.to_datetime(period["개설일"], format="%Y-%m-%d", errors="coerce")   # 개설일
  period["폐업일"] = pd.to_datetime(period["폐업일"], format="%Y-%m-%d", errors="coerce") # 폐업일

  유지기간_계산용 = period["폐업일"].fillna('2025-08-31')

  period["유지기간_일"] = (유지기간_계산용 - period["개설일"]).dt.days
  days_per_month = 365.25 / 12
  period['유지기간_월'] = (period['유지기간_일'] / days_per_month).round(1)

  period_result = period.drop_duplicates(subset=['가맹점구분번호'])
  create_features_result = create_features_result.merge(
      period_result[['가맹점구분번호', '유지기간_월']],
      on="가맹점구분번호",
      how="left")

  # ======================================================
  # 최근 3개월 매출 구간 평균(feature) 생성
  # ======================================================
  # 1) 가맹점별 최신 기준년월 계산
  #    - 각 가맹점 그룹의 최신 '기준년월'을 모든 행에 추가
  merged_1and2and3['가맹점_최근년월'] = merged_1and2and3.groupby('가맹점구분번호')['기준년월'].transform('max')

  # 2) 최근 3개월 데이터 필터링
  #    - '가맹점_최근년월' 기준, 현재 행이 최근 3개월(최신월 포함)인지 확인
  recent_3months  = merged_1and2and3[
      merged_1and2and3['기준년월'] >= (merged_1and2and3['가맹점_최근년월'] - pd.DateOffset(months=2))
  ].copy()

  # 3) 가맹점별 최근 3개월 평균 매출 계산
  recent_3months_avg  = recent_3months.groupby('가맹점구분번호')['매출금액 구간'].agg('mean').reset_index()

  # 4) 컬럼 이름 변경
  recent_3months_avg.rename(columns={
      '매출금액 구간': '최근 3개월 매출 평균',
  }, inplace=True)

  create_features_result = create_features_result.merge(
      recent_3months_avg,
      on='가맹점구분번호',
      how="left")

  # ======================================================
  # DATASET 2 파생변수 생성
  # ======================================================

  # 컬럼 선택
  normal_avg_cols = ["동일 업종 매출금액 비율"]
  temp_df = merged_1and2and3[["가맹점구분번호","기준년월"] + normal_avg_cols].copy()

  # 1) 변화량
  #     - 동일 업종 매출금액 비율
  temp_df = temp_df.sort_values(by=['가맹점구분번호', '기준년월'])
  temp_df["동일 업종 매출금액 비율_변화율"] = temp_df.groupby('가맹점구분번호')["동일 업종 매출금액 비율"].diff()

  # 3) 가맹점별 최종 정리
  temp_df.drop(columns=['기준년월', '동일 업종 매출금액 비율'],inplace=True)
  df_grouped = temp_df.groupby('가맹점구분번호').sum()

  # 병합
  create_features_result = pd.merge(create_features_result, df_grouped, on="가맹점구분번호", how="left")  # 변화량


  # ======================================================
  # DATASET 3 파생변수 생성
  # ======================================================

  # 가맹점별 평균 계산
  add_1 = merged_1and2and3.groupby("가맹점구분번호")["남성 50대 고객 비중"].mean().reset_index()

  create_features_result = create_features_result.merge(
      add_1,
      on='가맹점구분번호',
      how="left")

  # =========================
  # (NEW) 업종별 폐업률 생성
  # =========================
  # 업종별 폐업률 계산
  업종_폐업률 = (
      merged_1and2and3
      .drop_duplicates(subset='가맹점구분번호')[['가맹점구분번호', '업종', '폐업여부']]
      .groupby('업종')['폐업여부']
      .mean()
      .reset_index()
      .rename(columns={'폐업여부': '업종별_폐업률'})
  )

  # 가맹점에 업종별 폐업률 병합
  create_features_result = create_features_result.merge(
      merged_1and2and3[['가맹점구분번호', '업종']].drop_duplicates(),
      on='가맹점구분번호',
      how='left'
  )
  create_features_result = create_features_result.merge(
      업종_폐업률,
      on='업종',
      how='left'
  ).drop(columns='업종')

  return create_features_result

## feature selection

In [71]:
import numpy as np
import pandas as pd
from statsmodels.tools.tools import add_constant
from statsmodels.stats.outliers_influence import variance_inflation_factor

def clean_for_vif(X: pd.DataFrame, dropna_rows=True, verbose=True) -> pd.DataFrame:
    X = X.copy()

    # 1) inf -> NaN
    X = X.replace([np.inf, -np.inf], np.nan)

    # 2) 숫자형만
    X = X.select_dtypes(include=[np.number])

    # 3) 전부 NaN 컬럼 제거
    all_nan_cols = X.columns[X.isna().all()].tolist()
    if all_nan_cols and verbose:
        print("⚠️ [VIF] all-NaN cols drop:", all_nan_cols)
    X = X.drop(columns=all_nan_cols, errors="ignore")

    # 4) 상수열(분산 0) 제거
    const_cols = X.columns[X.nunique(dropna=True) <= 1].tolist()
    if const_cols and verbose:
        print("⚠️ [VIF] constant cols drop:", const_cols)
    X = X.drop(columns=const_cols, errors="ignore")

    # 5) 결측 포함 행 제거(권장: VIF는 결측 허용 안 함)
    if dropna_rows:
        na_rows = X.isna().any(axis=1).sum()
        if na_rows and verbose:
            print(f"⚠️ [VIF] drop rows with NaN: {na_rows}")
        X = X.dropna(axis=0)

    return X


In [72]:
def feature_selection(create_features_result):
    # -------------------------
    # 1) 변수 제거 전 VIF
    # -------------------------
    X = create_features_result.drop(columns=['가맹점구분번호', '폐업여부'])

    # ✅ 여기서 float 변환은 clean_for_vif 이후에 하는 게 안전
    X = clean_for_vif(X, dropna_rows=True, verbose=True)

    # (필요하면 float 변환)
    X = X.astype(float)

    X_const = add_constant(X, has_constant="add")

    vif = pd.DataFrame()
    vif["Feature"] = X_const.columns
    vif["VIF"] = [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]

    print("\n--- 변수 제거 전 분산팽창계수(VIF) ---")
    print(vif.sort_values("VIF", ascending=False))

    # -------------------------
    # 2) 변수 제거 후 VIF
    # -------------------------
    X2 = create_features_result.drop(columns=['가맹점구분번호','폐업여부','상권_집객력','상권_소비력','소비자 구매력'], errors="ignore")

    X2 = clean_for_vif(X2, dropna_rows=True, verbose=True)
    X2 = X2.astype(float)

    X2_const = add_constant(X2, has_constant="add")

    vif2 = pd.DataFrame()
    vif2["Feature"] = X2_const.columns
    vif2["VIF"] = [variance_inflation_factor(X2_const.values, i) for i in range(X2_const.shape[1])]

    print("\n--- 변수 제거 후 분산팽창계수(VIF) ---")
    print(vif2.sort_values("VIF", ascending=False))

    # -------------------------
    # 3) 최종 선택 변수 반환
    # -------------------------
    selected_cols = [
        '가맹점구분번호', '폐업여부', '상권_활성화지수',
        '물가 대비 민감도', '업종별_폐업률', '유지기간_월',
        '최근 3개월 매출 평균', '동일 업종 매출금액 비율_변화율',
        '남성 50대 고객 비중'
    ]

    # 혹시 없는 컬럼이 있어도 에러 안 나게
    existing = [c for c in selected_cols if c in create_features_result.columns]
    missing = [c for c in selected_cols if c not in create_features_result.columns]
    if missing:
        print("⚠️ 최종 선택 변수 중 데이터에 없는 컬럼:", missing)

    return create_features_result[existing]


# modeling

In [73]:
def modeling(features_selection_result):
  df = features_selection_result
  X = features_selection_result.drop(columns=['가맹점구분번호','폐업여부'])
  y = df["폐업여부"].astype(int)
  
# X, y 만든 직후
  if '상권_활성화지수' in X.columns:
      X['상권_활성화지수'] = X['상권_활성화지수'].fillna(X['상권_활성화지수'].median())

  print("NaN 총 개수:", X.isna().sum().sum())
  print("NaN 있는 컬럼:", X.columns[X.isna().any()].tolist())
  print("NaN 있는 행 수:", X.isna().any(axis=1).sum())
  print(X.columns)

  X = X.copy()
  X.columns = [f"f{i}" for i in range(X.shape[1])]

  # SMOTE + Tomek Links 적용
  smt = SMOTETomek(random_state=42)
  X, y = smt.fit_resample(X, y)  # X, y는 원래 데이터로 가정

  # 학습/테스트 데이터 분리
  X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

  # 상권_활성화지수 제외 후 표준화
  column_names = df.columns
  features_train = pd.DataFrame(X_train, columns=column_names).drop(columns=['상권_활성화지수'])
  features_test = pd.DataFrame(X_test, columns=column_names).drop(columns=['상권_활성화지수'])

  #  표준화
  scaler = StandardScaler()
  features_train_scaled = scaler.fit_transform(features_train)
  features_test_scaled = scaler.transform(features_test)


  scaler = StandardScaler()
  X_train_scaled = scaler.fit_transform(X_train)
  X_test_scaled = scaler.transform(X_test)

  # 모델 정의
  models = {
      'LogisticRegression': LogisticRegression(random_state=42, max_iter=1000),
      'SVM': SVC(random_state=42),
      'EasyEnsembleClassifier': EasyEnsembleClassifier(random_state=47, n_jobs=-1)
  }

  # 결과 저장
  results = {'Model': [], 'Accuracy': [], 'Precision': [], 'Recall': [], 'F1': []}

# ✅ 루프 전에 딱 1번만 변환
  if hasattr(X_train_scaled, "to_numpy"):
      X_train_scaled = X_train_scaled.to_numpy()
  if hasattr(X_test_scaled, "to_numpy"):
      X_test_scaled = X_test_scaled.to_numpy()

  X_train_scaled = np.asarray(X_train_scaled)
  X_test_scaled  = np.asarray(X_test_scaled)

  # ✅ y도 numpy로 통일 (선택)
  y_train = np.asarray(y_train)
  y_test  = np.asarray(y_test)

  # 모델별 학습 및 평가
  for model_name, model in models.items():
      model.fit(X_train_scaled, y_train)
      y_pred = model.predict(X_test_scaled)

      results['Model'].append(model_name)
      results['Accuracy'].append(accuracy_score(y_test, y_pred))
      results['Precision'].append(precision_score(y_test, y_pred))
      results['Recall'].append(recall_score(y_test, y_pred))
      results['F1'].append(f1_score(y_test, y_pred))

  # 결과를 DataFrame으로 변환
  results_df = pd.DataFrame(results)

  return results_df


# 하이퍼파라미터 튜닝

In [74]:
def hyperparameter_Tuning(features_selection_result):
  # =======================
  # 1) 데이터 로드 및 전처리
  # =======================
  df = features_selection_result

  X = features_selection_result.drop(columns=['가맹점구분번호','폐업여부'])
  y = df["폐업여부"].astype(int)

  # 표준화(상권_활성화지수 제외)
  cols = [c for c in X.columns if c != '상권_활성화지수']
  X_scaled = X.copy()
  scaler = StandardScaler()
  X_scaled[cols] = scaler.fit_transform(X[cols])

  # =======================
  # 2) 리샘플러 설정 (SMOTE + TomekLinks)
  # =======================
  # 소수 클래스(폐업=1)를 오버샘플링 + 경계 샘플 제거
  resampler = SMOTETomek(sampling_strategy=1.0, random_state=42)
  X_resampled, y_resampled = resampler.fit_resample(X_scaled, y)

  # =======================
  # 3) 하이퍼파라미터 그리드 설정 (EasyEnsembleClassifier 자체 파라미터만)
  # =======================
  param_grid = {
      'C': [0.1, 1, 10],
      'kernel': ['linear', 'rbf'],
      'gamma': ['scale', 'auto']
  }

  # =======================
  # 4) GridSearchCV 실행
  # =======================
  cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=47)


  # 여러 지표를 scorer로 설정
  scoring = {
      'f1': make_scorer(f1_score),
      'recall': make_scorer(recall_score),
      'precision': make_scorer(precision_score),
      'roc_auc': make_scorer(roc_auc_score, needs_proba=True)
  }

  model_svm = SVC(random_state=47, probability=True)

  grid_search = GridSearchCV(
      estimator=model_svm,
      param_grid=param_grid,
      cv=cv,
      scoring=scoring,
      refit='f1',
      n_jobs=-1,
      verbose=2,
      return_train_score=True
  )

  grid_search.fit(X_resampled, y_resampled)

  # =======================
  # 5) 최적 파라미터 및 모든 지표 출력
  # =======================
  print("\n" + "="*60)
  print("최적 하이퍼파라미터 및 성능 (GridSearchCV 결과)")
  print("="*60)
  print(f"▶ 최적 파라미터: {grid_search.best_params_}")

  # 최적 파라미터의 인덱스 찾기
  best_index = grid_search.best_index_

  # 모든 지표 출력
  print(f"\n▶ 최적 파라미터의 교차검증 성능:")
  print(f"  - F1 Score : {grid_search.cv_results_['mean_test_f1'][best_index]:.4f} (±{grid_search.cv_results_['std_test_f1'][best_index]:.4f})")
  print(f"  - Recall   : {grid_search.cv_results_['mean_test_recall'][best_index]:.4f} (±{grid_search.cv_results_['std_test_recall'][best_index]:.4f})")
  print(f"  - Precision: {grid_search.cv_results_['mean_test_precision'][best_index]:.4f} (±{grid_search.cv_results_['std_test_precision'][best_index]:.4f})")

  return grid_search

# 실행결과

In [75]:
# 데이터 로드
big_data_set1_f, big_data_set2_f, big_data_set3_f, gdf, unique_addresses, trade_area_info, consumer_price_index = data_load(path1, path2, path3, shp_path, address_xy_path, trade_area_path, consumer_price_index_path)
# 전처리
merged_1and2and3, address_to_district_map, seongdong_area, map = Preprocessing(big_data_set1_f, gdf, unique_addresses, big_data_set2_f, big_data_set3_f)
# # 지도 시각화 출력
# map


✅ encodings | ds1=cp949, ds2=cp949, ds3=utf-8, address=utf-8, trade=cp949, cpi=utf-8


In [76]:
# 파생변수 생성
create_features_result = create_features(path1, merged_1and2and3, trade_area_info, seongdong_area, consumer_price_index)
# 변수 선택
features_selection_result = feature_selection(create_features_result)
# 모델링
modeling_results=modeling(features_selection_result)
# 하이퍼파라미터 튜닝(그리드 서치)
grid_search = hyperparameter_Tuning(features_selection_result)

⚠️ [VIF] drop rows with NaN: 6

--- 변수 제거 전 분산팽창계수(VIF) ---
              Feature         VIF
0               const  183.645694
3            상권_활성화지수   30.080749
1              상권_집객력   21.900089
5             소비자 구매력    5.627311
7        최근 3개월 매출 평균    5.425834
2              상권_소비력    4.824469
9        남성 50대 고객 비중    1.165939
8   동일 업종 매출금액 비율_변화율    1.126492
6              유지기간_월    1.108520
4           물가 대비 민감도    1.025543
10            업종별_폐업률    1.017483
⚠️ [VIF] drop rows with NaN: 6

--- 변수 제거 후 분산팽창계수(VIF) ---
             Feature        VIF
0              const  19.888573
4       최근 3개월 매출 평균   1.124520
3             유지기간_월   1.099365
6       남성 50대 고객 비중   1.096292
5  동일 업종 매출금액 비율_변화율   1.080422
1           상권_활성화지수   1.061371
2          물가 대비 민감도   1.021878
7            업종별_폐업률   1.015430
NaN 총 개수: 0
NaN 있는 컬럼: []
NaN 있는 행 수: 0
Index(['상권_활성화지수', '물가 대비 민감도', '업종별_폐업률', '유지기간_월', '최근 3개월 매출 평균', '동일 업종 매출금액 비율_변화율', '남성 50대 고객 비중'], dtype='object')


UnicodeEncodeError: 'ascii' codec can't encode characters in position 18-20: ordinal not in range(128)

NameError: name 'X' is not defined

,가맹점구분번호,폐업여부,상권_활성화지수,물가 대비 민감도,업종별_폐업률,유지기간_월,최근 3개월 매출 평균,동일 업종 매출금액 비율_변화율,남성 50대 고객 비중
0,16184E93D9,False,1.268599,-20.930726,0.025641,149.4,3.333333,-39.2,17.584146
1,4D039EA8B7,False,1.268599,12.717065,0.025641,141.3,2.666667,-95.4,18.492058
2,0074C4990A,False,1.268599,55.003773,0.025641,135.7,2.0,-862.6,11.831754
3,68308F2746,False,1.268599,26.734986,0.025641,117.2,4.666667,14.0,7.111738
4,4117EDDE9C,False,1.268599,3.875,0.025641,116.7,5.666667,3.5,0.000000
...,...,...,...,...,...,...,...,...,...
3836,7FCF23E6F3,False,0.964572,0.821269,0.000000,24.3,4.0,42.8,5.173824
3837,9743B78531,False,0.964572,-9.955138,0.039216,149.9,1.333333,-3.6,8.270196
3838,C2261977A8,False,0.910491,1.867732,0.025641,27.5,1.666667,162.5,13.212625
3839,F1C69918DF,False,0.964572,4.591766,0.000000,46.8,3.333333,-29.6,8.254367
